<a href="https://colab.research.google.com/github/Heng1222/Ohsumed_classification/blob/feat-umap-projection/Model/Umap_projection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchinfo

## roberta-base Model


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaModel
from peft import PeftModel
from torchinfo import summary
import umap
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# 設定裝置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 定義分類模型
# ==========================================
class RobertaML(nn.Module):
    def __init__(self, model_path_or_name, freeze_backbone=True, useLoRA=False):
        super(RobertaML, self).__init__()
        self.roberta = RobertaModel.from_pretrained(model_path_or_name)

        # LoRA 掛載
        if useLoRA:
            LoRA_folder = "maxbeettww/roberta-MeSH-lora"
            self.roberta = PeftModel.from_pretrained(self.roberta, LoRA_folder)
            print("LoRA adapter loaded.")

        # 凍結 RoBERTa 參數
        if freeze_backbone:
            for param in self.roberta.parameters():
                param.requires_grad = False

        print("Model Summary...")
        print("base model：\n\n",summary(self.roberta))

    # 專門提取 Embedding
    def get_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output

# ==========================================
# 2. 資料集處理
# ==========================================
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors="pt")
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# ==========================================
# 3. 主程式邏輯
# ==========================================

# --- A. 讀取資料 ---
url = "https://media.githubusercontent.com/media/Heng1222/Ohsumed_classification/refs/heads/main/classification_data/ohsumed_dataset.csv"
print("Downloading and reading dataset...")
df_all = pd.read_csv(url)

# 抽2000筆測試
df_all = df_all.sample(3000, random_state=42).reset_index(drop=True)

# 處理 Label (將 C01 -> 0, C02 -> 1...)
label_encoder = LabelEncoder()
df_all['label_id'] = label_encoder.fit_transform(df_all['label'])
num_labels = len(label_encoder.classes_)
print(f"Unique classes: {num_labels}")

# --- B. 初始化模型與 Tokenizer ---
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Initializing Model...")
# 注意：這裡 useLoRA 設為 False，若您要測試 LoRA 版本請改為 True
model = RobertaML(model_name, freeze_backbone=True, useLoRA=False)
model.to(device)
model.eval()

# --- C. 準備 DataLoader ---
dataset = TextDataset(
    texts=df_all['title'].tolist(),
    labels=df_all['label_id'].tolist(),
    tokenizer=tokenizer,
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# --- D. 提取 768維 Embedding ---
print("Extracting Embeddings...")
embeddings_list = []
labels_list = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        emb = model.get_embeddings(input_ids, attention_mask)

        embeddings_list.append(emb.cpu().numpy())
        labels_list.extend(labels)

# 轉換為 numpy array (N, 768)
X_embeddings = np.vstack(embeddings_list)
y_true = np.array(labels_list)

print(f"Embedding shape: {X_embeddings.shape}")

# --- E. UMAP 降維 (降至 3維) ---
print("Running UMAP dimensionality reduction...")
umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
projections = umap_3d.fit_transform(X_embeddings)

# --- F. 3D 視覺化 (Plotly) ---
# 建立 DataFrame 方便後續繪圖
df_viz = pd.DataFrame(projections, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
df_viz['Label'] = label_encoder.inverse_transform(y_true) # 轉回 C01, C02 字串
df_viz['Title'] = df_all['title'] # 加入 Title 方便滑鼠移上去看

# ==========================================
# 4. Clustering 與 純度計算
# ==========================================

# 定義純度計算函數
def purity_score(y_true, y_pred):
    # 計算 Confusion Matrix
    contingency_matrix = confusion_matrix(y_true, y_pred)
    # 對於每個 Cluster，找出佔比最大的真實類別
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

print("\nRunning K-Means Clustering...")
# 我們已知有 num_labels 個類別，所以設 K = num_labels
kmeans = KMeans(n_clusters=num_labels, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_embeddings)

# 計算純度
purity = purity_score(y_true, y_pred)
print(f"--------------------------------------------------")
print(f"Clustering Purity Score: {purity:.4f}")
print(f"--------------------------------------------------")
print("說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。")
print("若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。")

Using device: cuda
Unique classes: 23


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Summary...
base model：

Layer (type:depth-idx)                                  Param #
RobertaModel                                            --
├─RobertaEmbeddings: 1-1                                --
│    └─Embedding: 2-1                                   (38,603,520)
│    └─Embedding: 2-2                                   (768)
│    └─LayerNorm: 2-3                                   (1,536)
│    └─Dropout: 2-4                                     --
│    └─Embedding: 2-5                                   (394,752)
├─RobertaEncoder: 1-2                                   --
│    └─ModuleList: 2-6                                  --
│    │    └─RobertaLayer: 3-1                           (7,087,872)
│    │    └─RobertaLayer: 3-2                           (7,087,872)
│    │    └─RobertaLayer: 3-3                           (7,087,872)
│    │    └─RobertaLayer: 3-4                           (7,087,872)
│    │    └─RobertaLayer: 3-5                           (7,087,872)
│    │    

100%|██████████| 94/94 [01:14<00:00,  1.27it/s]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedding shape: (3000, 768)
Running UMAP dimensionality reduction...

Running K-Means Clustering...
--------------------------------------------------
Clustering Purity Score: 0.2163
--------------------------------------------------
說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。
若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。


### Generating the 3D Plot

In [3]:
import plotly.express as px
from IPython.display import display
print("Generating 3D Plot...")
# Generate the plot using the existing df_viz DataFrame
fig = px.scatter_3d(
    df_viz,
    x='UMAP_1', y='UMAP_2', z='UMAP_3',
    color='Label',
    hover_data=['Title'],
    title='RoBERTa Base Embeddings of Ohsumed Titles (3D UMAP)',
    opacity=0.7,
    size_max=5
)

# Adjust layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
display(fig)

# You can also uncomment the line below to save the plot as an HTML file
# fig.write_html("roberta_embeddings_3d_umap_again.html")
# print("Plot saved as roberta_embeddings_3d_umap_again.html")

Generating 3D Plot...


## MLM Model


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaModel
from peft import PeftModel
from torchinfo import summary
import umap
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# 設定裝置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 定義分類模型
# ==========================================
class RobertaML(nn.Module):
    def __init__(self, model_path_or_name, freeze_backbone=True, useLoRA=False):
        super(RobertaML, self).__init__()
        self.roberta = RobertaModel.from_pretrained(model_path_or_name)

        # LoRA 掛載
        if useLoRA:
            LoRA_folder = "maxbeettww/roberta-MeSH-lora"
            self.roberta = PeftModel.from_pretrained(self.roberta, LoRA_folder)
            print("LoRA adapter loaded.")

        # 凍結 RoBERTa 參數
        if freeze_backbone:
            for param in self.roberta.parameters():
                param.requires_grad = False

        print("Model Summary...")
        print("base model：\n\n",summary(self.roberta))

    # 專門提取 Embedding
    def get_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output

# ==========================================
# 2. 資料集處理
# ==========================================
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors="pt")
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# ==========================================
# 3. 主程式邏輯
# ==========================================

# --- A. 讀取資料 ---
url = "https://media.githubusercontent.com/media/Heng1222/Ohsumed_classification/refs/heads/main/classification_data/ohsumed_dataset.csv"
print("Downloading and reading dataset...")
df_all = pd.read_csv(url)

# 抽2000筆測試
df_all = df_all.sample(3000, random_state=42).reset_index(drop=True)

# 處理 Label (將 C01 -> 0, C02 -> 1...)
label_encoder = LabelEncoder()
df_all['label_id'] = label_encoder.fit_transform(df_all['label'])
num_labels = len(label_encoder.classes_)
print(f"Unique classes: {num_labels}")

# --- B. 初始化模型與 Tokenizer ---
model_name = "maxbeettww/roberta-ohsumed-mlm"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Initializing Model...")
# 注意：這裡 useLoRA 設為 False，若您要測試 LoRA 版本請改為 True
model = RobertaML(model_name, freeze_backbone=True, useLoRA=False)
model.to(device)
model.eval()

# --- C. 準備 DataLoader ---
dataset = TextDataset(
    texts=df_all['title'].tolist(),
    labels=df_all['label_id'].tolist(),
    tokenizer=tokenizer,
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# --- D. 提取 768維 Embedding ---
print("Extracting Embeddings...")
embeddings_list = []
labels_list = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        emb = model.get_embeddings(input_ids, attention_mask)

        embeddings_list.append(emb.cpu().numpy())
        labels_list.extend(labels)

# 轉換為 numpy array (N, 768)
X_embeddings = np.vstack(embeddings_list)
y_true = np.array(labels_list)

print(f"Embedding shape: {X_embeddings.shape}")

# --- E. UMAP 降維 (降至 3維) ---
print("Running UMAP dimensionality reduction...")
umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
projections = umap_3d.fit_transform(X_embeddings)

# --- F. 3D 視覺化 (Plotly) ---
# 建立 DataFrame 方便後續繪圖
df_viz = pd.DataFrame(projections, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
df_viz['Label'] = label_encoder.inverse_transform(y_true) # 轉回 C01, C02 字串
df_viz['Title'] = df_all['title'] # 加入 Title 方便滑鼠移上去看

# ==========================================
# 4. Clustering 與 純度計算
# ==========================================

# 定義純度計算函數
def purity_score(y_true, y_pred):
    # 計算 Confusion Matrix
    contingency_matrix = confusion_matrix(y_true, y_pred)
    # 對於每個 Cluster，找出佔比最大的真實類別
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

print("\nRunning K-Means Clustering...")
# 我們已知有 num_labels 個類別，所以設 K = num_labels
kmeans = KMeans(n_clusters=num_labels, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_embeddings)

# 計算純度
purity = purity_score(y_true, y_pred)
print(f"--------------------------------------------------")
print(f"Clustering Purity Score: {purity:.4f}")
print(f"--------------------------------------------------")
print("說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。")
print("若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。")

Using device: cuda
Unique classes: 23


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Initializing Model...


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: maxbeettww/roberta-ohsumed-mlm
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Summary...
base model：

Layer (type:depth-idx)                                  Param #
RobertaModel                                            --
├─RobertaEmbeddings: 1-1                                --
│    └─Embedding: 2-1                                   (38,603,520)
│    └─Embedding: 2-2                                   (768)
│    └─LayerNorm: 2-3                                   (1,536)
│    └─Dropout: 2-4                                     --
│    └─Embedding: 2-5                                   (394,752)
├─RobertaEncoder: 1-2                                   --
│    └─ModuleList: 2-6                                  --
│    │    └─RobertaLayer: 3-1                           (7,087,872)
│    │    └─RobertaLayer: 3-2                           (7,087,872)
│    │    └─RobertaLayer: 3-3                           (7,087,872)
│    │    └─RobertaLayer: 3-4                           (7,087,872)
│    │    └─RobertaLayer: 3-5                           (7,087,872)
│    │    

100%|██████████| 94/94 [01:21<00:00,  1.15it/s]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Embedding shape: (3000, 768)
Running UMAP dimensionality reduction...

Running K-Means Clustering...
--------------------------------------------------
Clustering Purity Score: 0.2573
--------------------------------------------------
說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。
若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。


### Generating the 3D Plot

In [5]:
import plotly.express as px
from IPython.display import display
print("Generating 3D Plot...")
# Generate the plot using the existing df_viz DataFrame
fig = px.scatter_3d(
    df_viz,
    x='UMAP_1', y='UMAP_2', z='UMAP_3',
    color='Label',
    hover_data=['Title'],
    title='MLM Model Embeddings of Ohsumed Titles (3D UMAP)',
    opacity=0.7,
    size_max=5
)

# Adjust layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
display(fig)

# You can also uncomment the line below to save the plot as an HTML file
# fig.write_html("roberta_embeddings_3d_umap_again.html")
# print("Plot saved as roberta_embeddings_3d_umap_again.html")

Generating 3D Plot...


## MeSH LoRA Model


In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaModel
from peft import PeftModel
from torchinfo import summary
import umap
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# 設定裝置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 定義分類模型
# ==========================================
class RobertaML(nn.Module):
    def __init__(self, model_path_or_name, freeze_backbone=True, useLoRA=False):
        super(RobertaML, self).__init__()
        self.roberta = RobertaModel.from_pretrained(model_path_or_name)

        # LoRA 掛載
        if useLoRA:
            LoRA_folder = "maxbeettww/roberta-MeSH-lora"
            self.roberta = PeftModel.from_pretrained(self.roberta, LoRA_folder)
            print("LoRA adapter loaded.")

        # 凍結 RoBERTa 參數
        if freeze_backbone:
            for param in self.roberta.parameters():
                param.requires_grad = False

        print("Model Summary...")
        print("base model：\n\n",summary(self.roberta))

    # 專門提取 Embedding
    def get_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :]
        return cls_output

# ==========================================
# 2. 資料集處理
# ==========================================
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors="pt")
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# ==========================================
# 3. 主程式邏輯
# ==========================================

# --- A. 讀取資料 ---
url = "https://media.githubusercontent.com/media/Heng1222/Ohsumed_classification/refs/heads/main/classification_data/ohsumed_dataset.csv"
print("Downloading and reading dataset...")
df_all = pd.read_csv(url)

# 抽2000筆測試
df_all = df_all.sample(3000, random_state=42).reset_index(drop=True)

# 處理 Label (將 C01 -> 0, C02 -> 1...)
label_encoder = LabelEncoder()
df_all['label_id'] = label_encoder.fit_transform(df_all['label'])
num_labels = len(label_encoder.classes_)
print(f"Unique classes: {num_labels}")

# --- B. 初始化模型與 Tokenizer ---
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Initializing Model...")
# 注意：這裡 useLoRA 設為 False，若您要測試 LoRA 版本請改為 True
model = RobertaML(model_name, freeze_backbone=True, useLoRA=True)
model.to(device)
model.eval()

# --- C. 準備 DataLoader ---
dataset = TextDataset(
    texts=df_all['title'].tolist(),
    labels=df_all['label_id'].tolist(),
    tokenizer=tokenizer,
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# --- D. 提取 768維 Embedding ---
print("Extracting Embeddings...")
embeddings_list = []
labels_list = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        emb = model.get_embeddings(input_ids, attention_mask)

        embeddings_list.append(emb.cpu().numpy())
        labels_list.extend(labels)

# 轉換為 numpy array (N, 768)
X_embeddings = np.vstack(embeddings_list)
y_true = np.array(labels_list)

print(f"Embedding shape: {X_embeddings.shape}")

# --- E. UMAP 降維 (降至 3維) ---
print("Running UMAP dimensionality reduction...")
umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
projections = umap_3d.fit_transform(X_embeddings)

# --- F. 3D 視覺化 (Plotly) ---
# 建立 DataFrame 方便後續繪圖
df_viz = pd.DataFrame(projections, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
df_viz['Label'] = label_encoder.inverse_transform(y_true) # 轉回 C01, C02 字串
df_viz['Title'] = df_all['title'] # 加入 Title 方便滑鼠移上去看

# ==========================================
# 4. Clustering 與 純度計算
# ==========================================

# 定義純度計算函數
def purity_score(y_true, y_pred):
    # 計算 Confusion Matrix
    contingency_matrix = confusion_matrix(y_true, y_pred)
    # 對於每個 Cluster，找出佔比最大的真實類別
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

print("\nRunning K-Means Clustering...")
# 我們已知有 num_labels 個類別，所以設 K = num_labels
kmeans = KMeans(n_clusters=num_labels, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_embeddings)

# 計算純度
purity = purity_score(y_true, y_pred)
print(f"--------------------------------------------------")
print(f"Clustering Purity Score: {purity:.4f}")
print(f"--------------------------------------------------")
print("說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。")
print("若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。")

Using device: cuda
Unique classes: 23
Initializing Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/7.68M [00:00<?, ?B/s]

LoRA adapter loaded.
Model Summary...
base model：

Layer (type:depth-idx)                                                 Param #
PeftModel                                                              --
├─LoraModel: 1-1                                                       --
│    └─RobertaModel: 2-1                                               --
│    │    └─RobertaEmbeddings: 3-1                                     (39,000,576)
│    │    └─RobertaEncoder: 3-2                                        (86,971,392)
│    │    └─RobertaPooler: 3-3                                         (590,592)
Total params: 126,562,560
Trainable params: 0
Non-trainable params: 126,562,560
Extracting Embeddings...


100%|██████████| 94/94 [01:31<00:00,  1.03it/s]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Embedding shape: (3000, 768)
Running UMAP dimensionality reduction...

Running K-Means Clustering...
--------------------------------------------------
Clustering Purity Score: 0.2753
--------------------------------------------------
說明：Purity 越高 (接近 1.0) 代表模型 Embedding 空間中的群聚與真實標籤越一致。
若 Purity 很低 (例如 < 0.2)，代表 RoBERTa 在未經訓練下無法有效區分這些醫學類別。


### Generating the 3D Plot

In [7]:
import plotly.express as px
from IPython.display import display
print("Generating 3D Plot...")
# Generate the plot using the existing df_viz DataFrame
fig = px.scatter_3d(
    df_viz,
    x='UMAP_1', y='UMAP_2', z='UMAP_3',
    color='Label',
    hover_data=['Title'],
    title='MeSH LoRA Model Embeddings of Ohsumed Titles (3D UMAP)',
    opacity=0.7,
    size_max=5
)

# Adjust layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
display(fig)

# You can also uncomment the line below to save the plot as an HTML file
# fig.write_html("roberta_embeddings_3d_umap_again.html")
# print("Plot saved as roberta_embeddings_3d_umap_again.html")

Generating 3D Plot...
